# LangGraph: Stateful, Multi-Step & Cyclical Agent Workflows

**Workflow chosen:** A Research Assistant that **plans → retrieves → drafts → critiques → formats → sends** an answer, with a self-correction loop and a human-approval gate before the "risky" send action.

The workflow demonstrates LangGraph's **stateful graph execution, conditional routing, cyclical self-correction, human-in-the-loop interrupts, and checkpoint-based persistence**.

The LLM uses Groq's OpenAI-compatible endpoint, continuing the tooling approach from Day 2. Configure `GROQ_API_KEY` through the notebook's environment/secrets before running.

## Task 1: Graph Concepts & State Design

### Core building blocks

| Concept | What it is |
|---|---|
| **State** | A shared `TypedDict`/Pydantic model that carries information between nodes. Each node reads the current state and returns a partial update that LangGraph merges into the graph state. |
| **Node** | A Python function that receives the current state and returns a state update. A node can perform one unit of work, such as an LLM call, tool call, retrieval step, or calculation. |
| **Edge** | A fixed transition from one node to another, defining the normal flow of execution. |
| **Conditional Edge** | A routing mechanism that evaluates the current state and selects the next node dynamically. It enables branching and loop-back behavior. |
| **StateGraph** | The graph builder used to define nodes, edges, and conditional edges before compiling the workflow into a runnable graph. |

### State schema

The workflow state includes `search_plan` as a dedicated field so that the output produced by the `plan` node is explicitly stored in the shared state and consumed by the `retrieve` node.

Other fields track the retrieved results, draft, critique, quality score, retry count, human approval, final answer, and execution log.

In [1]:
from typing import TypedDict, List, Optional
from typing_extensions import Annotated
import operator


class AgentState(TypedDict):
    # Original user request
    query: str

    # Search query generated by the planning node
    search_plan: str

    # Results returned by the retrieval node
    search_results: List[str]

    # Current answer draft
    draft: str

    # Feedback produced by the critique node
    critique: str

    # Quality score assigned by the critique node (0-10)
    quality_score: int

    # Number of regeneration attempts after the initial draft
    retry_count: int

    # Maximum number of allowed regenerations
    max_retries: int

    # Final formatted answer
    final_answer: str

    # Payload prepared for the simulated risky action
    email_draft: str

    # Human decision at the approval gate
    human_approved: Optional[bool]

    # Append-only execution history
    log: Annotated[List[str], operator.add]

### Graph diagram (before coding)

The workflow follows a linear path initially, with a conditional self-correction loop after `critique`. If the quality score is below the threshold and retries remain, the graph returns to `generate`. Otherwise, it proceeds to `format` and pauses for human approval before the `send_email` action.

```text
            ┌─────────┐
            │  plan   │
            └────┬────┘
                 │
                 ▼
            ┌─────────┐
            │ retrieve│
            └────┬────┘
                 │
                 ▼
            ┌─────────┐
            │ generate│
            └────┬────┘
                 │
                 ▼
            ┌─────────┐
            │ critique│
            └────┬────┘
                 │
        ┌────────┴─────────┐
        │                  │
 quality low           quality OK
 & retries left        OR retries exhausted
        │                  │
        ▼                  ▼
   ┌─────────┐        ┌─────────┐
   │ generate│        │  format │
   └─────────┘        └────┬────┘
        ▲                   │
        │                   ▼
        │             ⏸ HUMAN APPROVAL
        │                   │
        │                   ▼
        │             ┌────────────┐
        │             │ send_email │
        │             └─────┬──────┘
        │                   │
        │                   ▼
        └───────────────►  END

In [2]:
%pip install -U langchain-groq

In [3]:
%pip install -U tavily-python

## Groq client + LLM call helper

In [4]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

# Load environment variables
load_dotenv()

# Get API key
api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError(
        "GROQ_API_KEY not found. Please add it to your .env file."
    )

# Initialize LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=api_key,
    temperature=0
)

print("API key loaded:", bool(api_key))
print("Setup complete and model initialized!")

API key loaded: True
Setup complete and model initialized!


In [5]:
from tavily import TavilyClient

tavily_api_key = os.getenv("TAVILY_API_KEY")

if not tavily_api_key:
    raise ValueError(
        "TAVILY_API_KEY not found. Please add it to your .env file."
    )

tavily_client = TavilyClient(api_key=tavily_api_key)

print("Tavily search client initialized successfully!")

Tavily search client initialized successfully!


In [6]:
SYSTEM_PROMPTS = {

    "plan": (
        "Turn the user's question into one concise, specific web search query. "
        "Return only the search query. Do not answer the question."
    ),

    "generate": (
        "Write a concise draft answer using only the provided web search results. "
        "Do not invent facts or use unsupported claims. "
        "If the search results do not contain enough information, say so. "
        "Write 3-5 clear sentences."
    ),

    "critique": (
        "Evaluate the draft answer for correctness, relevance, completeness, "
        "and whether its claims are supported by the provided search results. "
        "Give a score from 0 to 10.\n\n"

        "You MUST respond in exactly this format:\n"
        "SCORE: <number>/10\n"
        "<one concise line of feedback>\n"
        "<NEEDS_REVISION or OK>"
    ),

    "format": (
        "Produce a clean, concise final answer from the approved draft. "
        "Use only information supported by the draft and search results. "
        "Do not introduce new unsupported claims."
    ),
}


def llm_call(tag: str, user_content: str) -> str:
    """Call the Groq LLM using the prompt associated with a graph node."""

    if tag not in SYSTEM_PROMPTS:
        raise ValueError(
            f"Unknown prompt tag: {tag}. "
            f"Available tags: {list(SYSTEM_PROMPTS.keys())}"
        )

    response = llm.invoke([
        ("system", SYSTEM_PROMPTS[tag]),
        ("user", user_content),
    ])

    return response.content.strip()

## Task 2: Build a Linear Graph

We first build a simple linear LangGraph workflow:

`plan → retrieve → generate → format`

At this stage, there are **no conditional edges or cycles**. Each node executes exactly once in a fixed sequence.

The graph is compiled and run on a sample query, and the state is printed after each node to verify that each step produces the expected state updates.

In [7]:
SEARCH_DEPTH = "basic"
MAX_SEARCH_RESULTS = 3

In [8]:
from langgraph.graph import StateGraph, END


# ============================================================
# 1. PLAN NODE
# ============================================================

def plan_node(state: AgentState) -> dict:

    search_plan = llm_call(
        "plan",
        state["query"]
    )

    return {
        "search_plan": search_plan,
        "log": [f"[plan] {search_plan}"]
    }


# ============================================================
# 2. RETRIEVE NODE
# ============================================================

def retrieve_node(state: AgentState) -> dict:
    search_query = state["search_plan"]

    print(f"[retrieve] Searching web for: {search_query}")

    response = tavily_client.search(
        query=search_query,
        search_depth=SEARCH_DEPTH,
        max_results=MAX_SEARCH_RESULTS,
        include_answer=False
    )

    results = response.get("results", [])

    if not results:
        return {
            "search_results": [],
            "log": ["[retrieve] No results found."]
        }

    formatted_results = []

    for i, result in enumerate(results, start=1):
        title = result.get("title", "")
        content = result.get("content", "")
        url = result.get("url", "")

        formatted_results.append(
            f"{i}. {title}\n"
            f"   {content}\n"
            f"   Source: {url}"
        )

    print(f"[retrieve] Retrieved {len(formatted_results)} results.")

    return {
        "search_results": formatted_results,
        "log": [
            f"[retrieve] Retrieved {len(formatted_results)} web results."
        ]
    }

# ============================================================
# 3. GENERATE NODE
# ============================================================

def generate_node(state: AgentState) -> dict:

    search_results_text = "\n\n".join(
        state["search_results"]
    )

    prompt = (
        f"Query:\n{state['query']}\n\n"
        f"Web search results:\n{search_results_text}"
    )

    # Regeneration: provide previous draft and critique.
    if state["draft"]:
        prompt += (
            f"\n\nPrevious draft:\n{state['draft']}"
            f"\n\nPrevious critique:\n{state['critique']}"
        )

    draft = llm_call(
        "generate",
        prompt
    )

    if state["draft"]:
        retry_count = state["retry_count"] + 1
        generation_type = "regeneration"
    else:
        retry_count = 0
        generation_type = "initial generation"

    return {
        "draft": draft,
        "retry_count": retry_count,
        "log": [
            f"[generate] {generation_type} completed; "
            f"retries_used={retry_count}"
        ],
    }


# ============================================================
# 4. FORMAT NODE
# ============================================================

def format_node(state: AgentState) -> dict:

    prompt = (
        f"Original query:\n{state['query']}\n\n"
        f"Draft:\n{state['draft']}\n\n"
        f"Critique:\n{state['critique']}\n\n"
        f"Quality score: {state['quality_score']}/10"
    )

    final = llm_call(
        "format",
        prompt
    )

    return {
        "final_answer": final,
        "log": ["[format] final answer assembled"]
    }


# ============================================================
# 5. BUILD LINEAR GRAPH
# ============================================================

linear_builder = StateGraph(AgentState)

linear_builder.add_node("plan", plan_node)
linear_builder.add_node("retrieve", retrieve_node)
linear_builder.add_node("generate", generate_node)
linear_builder.add_node("format", format_node)

linear_builder.set_entry_point("plan")

linear_builder.add_edge("plan", "retrieve")
linear_builder.add_edge("retrieve", "generate")
linear_builder.add_edge("generate", "format")
linear_builder.add_edge("format", END)

linear_graph = linear_builder.compile()

print("Linear LangGraph compiled successfully!")

Linear LangGraph compiled successfully!


In [9]:
init_state = {
    "query": "What are the key ideas behind LangGraph?",
    "search_plan": "",
    "search_results": [],
    "draft": "",
    "critique": "",
    "quality_score": 0,
    "retry_count": 0,
    "max_retries": 2,
    "final_answer": "",
    "email_draft": "",
    "human_approved": None,
    "log": [],
}


print("===== STREAMING LINEAR GRAPH =====\n")

final_state = None

for step, update in enumerate(
    linear_graph.stream(init_state, stream_mode="values"),
    start=1
):
    final_state = update

    print(f"\n{'=' * 60}")
    print(f"STATE UPDATE {step}")
    print(f"{'=' * 60}")

    print("Query:")
    print(update["query"])

    print("\nSearch plan:")
    print(update["search_plan"])

    print("\nNumber of search results:")
    print(len(update["search_results"]))

    print("\nDraft:")
    print(update["draft"])

    print("\nFinal answer:")
    print(update["final_answer"])

    print("\nLatest log:")
    if update["log"]:
        print(update["log"][-1])
    else:
        print("(no log yet)")

print("\n\n===== FINAL RESULT =====")

if final_state is not None:
    print("\nFinal answer:")
    print(final_state["final_answer"])

    print("\nExecution log:")
    for entry in final_state["log"]:
        print(entry)

===== STREAMING LINEAR GRAPH =====


STATE UPDATE 1
Query:
What are the key ideas behind LangGraph?

Search plan:


Number of search results:
0

Draft:


Final answer:


Latest log:
(no log yet)

STATE UPDATE 2
Query:
What are the key ideas behind LangGraph?

Search plan:
"LangGraph key concepts"

Number of search results:
0

Draft:


Final answer:


Latest log:
[plan] "LangGraph key concepts"
[retrieve] Searching web for: "LangGraph key concepts"
[retrieve] Retrieved 3 results.

STATE UPDATE 3
Query:
What are the key ideas behind LangGraph?

Search plan:
"LangGraph key concepts"

Number of search results:
3

Draft:


Final answer:


Latest log:
[retrieve] Retrieved 3 web results.

STATE UPDATE 4
Query:
What are the key ideas behind LangGraph?

Search plan:
"LangGraph key concepts"

Number of search results:
3

Draft:
The key ideas behind LangGraph include the concept of state, which is a shared data structure representing the current snapshot of an application. Nodes in LangGraph comm

## Task 3: Conditional Edges & Cycles (Self-Correction Loop)

The `critique` node evaluates the generated draft and assigns a quality score.

A conditional edge then determines the next step:

- If the quality score is below the threshold **and** retries remain, the graph loops back to `generate` for regeneration.
- If the quality score meets the threshold **or** the maximum retry limit has been reached, the graph proceeds to `format`.

The `retry_count` and `max_retries` fields in the shared state prevent the self-correction loop from running indefinitely. Each generation and critique pass is recorded in the execution log so the retry behavior can be inspected.

In [10]:
import re

# Demo mode: force one retry so we can verify the cycle works
FORCE_RETRY_DEMO = True


def critique_node(state: AgentState) -> dict:
    search_results_text = "\n\n".join(state["search_results"])

    prompt = (
        f"Query: {state['query']}\n\n"
        f"Draft:\n{state['draft']}\n\n"
        f"Search Results:\n{search_results_text}"
    )

    # Demo mode: force the first critique to fail
    if FORCE_RETRY_DEMO and state["retry_count"] == 0:
        feedback = "SCORE: 5/10\nThe draft needs improvement."
    else:
        feedback = llm_call("critique", prompt)

    score_match = re.search(
        r"SCORE:\s*(\d+)(?:\s*/\s*10)?",
        feedback,
        re.IGNORECASE
    )

    if score_match:
        score = int(score_match.group(1))
        score = max(0, min(score, 10))
    else:
        score = 0

    return {
        "critique": feedback,
        "quality_score": score,
        "log": [
            f"[critique] score={score}, "
            f"retries_used={state['retry_count']}"
        ],
    }


QUALITY_THRESHOLD = 7


def route_after_critique(state: AgentState) -> str:

    # Good enough → move forward
    if state["quality_score"] >= QUALITY_THRESHOLD:
        return "format"

    # Maximum regeneration attempts reached
    if state["retry_count"] >= state["max_retries"]:
        return "format"

    # Not good enough → regenerate
    return "generate"


loop_builder = StateGraph(AgentState)

loop_builder.add_node("plan", plan_node)
loop_builder.add_node("retrieve", retrieve_node)
loop_builder.add_node("generate", generate_node)
loop_builder.add_node("critique", critique_node)
loop_builder.add_node("format", format_node)

loop_builder.set_entry_point("plan")

loop_builder.add_edge("plan", "retrieve")
loop_builder.add_edge("retrieve", "generate")
loop_builder.add_edge("generate", "critique")

loop_builder.add_conditional_edges(
    "critique",
    route_after_critique,
    {"generate": "generate", "format": "format"},
)

loop_builder.add_edge("format", END)

loop_graph = loop_builder.compile()

In [12]:
loop_state = {
    **init_state,
    "query": "Explain why self-correction loops need a retry cap."
}

print("--- Running the self-correcting graph ---\n")

result = loop_graph.invoke(loop_state)

for line in result["log"]:
    print(line)

print("\nTotal retries:", result["retry_count"])
print("Final quality score:", result["quality_score"])
print("\nFinal answer:")
print(result["final_answer"])

--- Running the self-correcting graph ---

[retrieve] Searching web for: "self-correction loops retry cap prevention"
[retrieve] Retrieved 3 results.
[plan] "self-correction loops retry cap prevention"
[retrieve] Retrieved 3 web results.
[generate] initial generation completed; retries_used=0
[critique] score=5, retries_used=0
[generate] regeneration completed; retries_used=1
[critique] score=9, retries_used=1
[format] final answer assembled

Total retries: 1
Final quality score: 9

Final answer:
A self-correction loop in AI systems needs a retry cap to prevent infinite loops when the system cannot correct its errors. This cap, also known as a "retry budget," is a configurable maximum number of retries that mitigates the risk of infinite loops. Without it, the system could retry indefinitely, wasting resources. For example, setting a default retry limit, such as 3, prevents the system from spinning on unfixable errors.


**Why this is awkward in a plain** **`AgentExecutor`** **but natural in LangGraph:**

`AgentExecutor` is primarily designed around an agent execution loop, where the model reasons, optionally calls tools, observes the results, and continues until the agent finishes or execution limits are reached. Expressing a workflow-specific retry such as "go back specifically to the drafting step, evaluate the result again, and allow only N regeneration attempts" is less explicit.

LangGraph makes this pattern a first-class **edge in the graph**. The retry boundary, quality condition, and retry cap (`max_retries` in `State`) are explicit and inspectable, making the self-correction workflow easier to control, debug, and extend.

## Task 4: Human-in-the-Loop & Interrupts

A `send_email` node represents a **risky external action**. Using
`interrupt_before=["send_email"]`, the graph pauses immediately before
executing the email-sending node, giving a human the opportunity to
approve or reject the action.

The graph uses a checkpointer (`MemorySaver`) so the paused state can be
persisted and resumed after the human decision. In this notebook, both
paths are demonstrated:

- **Reject:** `human_approved = False` → the email is not sent.
- **Approve:** `human_approved = True` → the `send_email` node executes.

This pattern is useful when an agent can prepare an external action
autonomously but should require human approval before actually performing
it.

In [13]:
# ============================================================
# HUMAN-IN-THE-LOOP
# ============================================================

def format_node_v2(state: AgentState) -> dict:
    prompt = (
        f"Draft:\n{state['draft']}\n\n"
        f"Critique:\n{state['critique']}"
    )

    final = llm_call("format", prompt)

    email_body = (
        "Subject: Research summary\n\n"
        f"{final}"
    )

    return {
        "final_answer": final,
        "email_draft": email_body,
        "log": [
            "[format] final answer + email draft assembled"
        ],
    }


def send_email_node(state: AgentState) -> dict:
    if state["human_approved"] is True:
        return {
            "log": [
                f"[send_email] APPROVED -> sent:\n"
                f"{state['email_draft']}"
            ]
        }

    return {
        "log": [
            "[send_email] REJECTED -> action cancelled, nothing sent"
        ]
    }


# ============================================================
# BUILD HITL GRAPH
# ============================================================

from langgraph.checkpoint.memory import MemorySaver

hitl_builder = StateGraph(AgentState)

hitl_builder.add_node("plan", plan_node)
hitl_builder.add_node("retrieve", retrieve_node)
hitl_builder.add_node("generate", generate_node)
hitl_builder.add_node("critique", critique_node)
hitl_builder.add_node("format", format_node_v2)
hitl_builder.add_node("send_email", send_email_node)

hitl_builder.set_entry_point("plan")

hitl_builder.add_edge("plan", "retrieve")
hitl_builder.add_edge("retrieve", "generate")
hitl_builder.add_edge("generate", "critique")

hitl_builder.add_conditional_edges(
    "critique",
    route_after_critique,
    {
        "generate": "generate",
        "format": "format",
    },
)

hitl_builder.add_edge("format", "send_email")
hitl_builder.add_edge("send_email", END)


# ============================================================
# PERSISTENCE + INTERRUPT
# ============================================================

checkpointer = MemorySaver()

hitl_graph = hitl_builder.compile(
    checkpointer=checkpointer,
    interrupt_before=["send_email"],
)

print("HITL graph compiled successfully!")

HITL graph compiled successfully!


In [15]:
# ============================================================
# HITL TEST — REJECTION
# ============================================================

config = {
    "configurable": {
        "thread_id": "demo-thread-1"
    }
}

hitl_state = {
    **init_state,
    "query": "Summarize this week's agent-framework progress."
}

print("--- Running until the interrupt ---\n")

for update in hitl_graph.stream(
    hitl_state,
    config,
    stream_mode="values"
):
    pass


# Inspect persisted state at the interrupt
paused_state = hitl_graph.get_state(config)

print("Graph paused.")
print("Next node queued:", paused_state.next)

print("\nEmail drafted, awaiting approval:")
print(paused_state.values["email_draft"])


# ------------------------------------------------------------
# Human rejects
# ------------------------------------------------------------

print("\n--- Human decision: REJECT ---")

hitl_graph.update_state(
    config,
    {"human_approved": False}
)

result_rejected = hitl_graph.invoke(
    None,
    config
)

print("\nExecution result:")
for entry in result_rejected["log"][-2:]:
    print(entry)

--- Running until the interrupt ---

[retrieve] Searching web for: "latest agent framework updates this week"
[retrieve] Retrieved 3 results.
Graph paused.
Next node queued: ('send_email',)

Email drafted, awaiting approval:
Subject: Research summary

The agent-framework has made significant progress, including promoting the agent-framework-ag-ui package to stable, adding security guidance for custom MCP Streamable HTTP clients, and fixing issues with stateless replay and Gemini 3 thought signatures. The orchestration layer has reached 1.0, providing stable support for coordination patterns across Python and .NET. New features such as FastAPI SSE keepalive support and forwarding skill directories to the Copilot session have also been added, aiming to improve the framework's stability and functionality.

--- Human decision: REJECT ---

Execution result:
[format] final answer + email draft assembled
[send_email] REJECTED -> action cancelled, nothing sent


In [16]:

# HITL TEST — APPROVAL


config2 = {
    "configurable": {
        "thread_id": "demo-thread-2"
    }
}

hitl_state2 = {
    **init_state,
    "query": "Draft a note about today's LangGraph exercise."
}

print("\n\n--- Running second thread until interrupt ---\n")

for update in hitl_graph.stream(
    hitl_state2,
    config2,
    stream_mode="values"
):
    pass


paused_state2 = hitl_graph.get_state(config2)

print("Graph paused.")
print("Next node queued:", paused_state2.next)

print("\nEmail drafted:")
print(paused_state2.values["email_draft"])


# ------------------------------------------------------------
# Human approves
# ------------------------------------------------------------

print("\n--- Human decision: APPROVE ---")

hitl_graph.update_state(
    config2,
    {"human_approved": True}
)

result_approved = hitl_graph.invoke(
    None,
    config2
)

print("\nExecution result:")
for entry in result_approved["log"][-2:]:
    print(entry)



--- Running second thread until interrupt ---

[retrieve] Searching web for: "LangGraph exercise summary today"
[retrieve] Retrieved 3 results.
Graph paused.
Next node queued: ('send_email',)

Email drafted:
Subject: Research summary

The LangGraph exercise involved creating a personalized compliment agent by setting up a node, defining an agent state, and constructing a graph to process user input. This foundational step reinforces understanding of nodes, state updates, and graph execution, preparing users to build more advanced applications in LangGraph.

--- Human decision: APPROVE ---

Execution result:
[format] final answer + email draft assembled
[send_email] APPROVED -> sent:
Subject: Research summary

The LangGraph exercise involved creating a personalized compliment agent by setting up a node, defining an agent state, and constructing a graph to process user input. This foundational step reinforces understanding of nodes, state updates, and graph execution, preparing users t

**When should a real product require human-in-the-loop vs. allow full autonomy?**

The decision should primarily depend on **reversibility and blast radius**. Actions that spend money, send external communications (such as emails or messages), delete or overwrite data, or act on another person's behalf should generally require human approval, especially when the agent is new and trust has not yet been established.

Full autonomy is more appropriate for actions that are **read-only, low-risk, easy to undo, sandboxed, or well-validated through production history**. Examples include searching for information, drafting content without sending it, or writing to a temporary workspace that a user can review later.

## Task 5: Persistence, Time-Travel & Framework Comparison

`MemorySaver` provides checkpoint-based persistence for the graph state,
keyed by `thread_id`. Below, we demonstrate:

1. **Persistence & resuming:** inspecting the saved state of a completed
   or paused thread and resuming execution from a checkpoint.

2. **State history & time-travel debugging:** walking through the
   checkpoints of a thread and selecting an earlier checkpoint to
   replay/resume the workflow from that point.

> **Note:** `MemorySaver` stores checkpoints in memory, so this
> demonstration persists state during the current Python process.
> Production applications that need persistence across process restarts
> should use a durable checkpointer.

In [17]:
# ------------------------------------------------------------
# 1. RESUME / PERSISTED STATE
# ------------------------------------------------------------

resumed_state = hitl_graph.get_state(config)

print("===== PERSISTED THREAD STATE =====")
print("Thread ID: demo-thread-1")

print("\nNext node:")
print(resumed_state.next)

print("\nRetry count:")
print(resumed_state.values["retry_count"])

print("\nHuman approval:")
print(resumed_state.values["human_approved"])

print("\nFinal answer:")
print(resumed_state.values["final_answer"])

print("\nLatest log entry:")
print(resumed_state.values["log"][-1])


# ------------------------------------------------------------
# 2. STATE HISTORY
# ------------------------------------------------------------

print("\n\n===== CHECKPOINT HISTORY =====")

history = list(hitl_graph.get_state_history(config2))

for i, snapshot in enumerate(reversed(history)):

    logs = snapshot.values.get("log", [])

    latest_log = (
        logs[-1]
        if logs
        else "(no log yet)"
    )

    print(
        f"Step {i}: "
        f"next={snapshot.next} | "
        f"retry_count={snapshot.values.get('retry_count')} | "
        f"quality_score={snapshot.values.get('quality_score')} | "
        f"{latest_log}"
    )

# Select an earlier checkpoint
earlier_checkpoint = history[len(history) // 2]

print("\n===== SELECTED CHECKPOINT =====")
print("Next node:")
print(earlier_checkpoint.next)
print("Retry count:")
print(earlier_checkpoint.values.get("retry_count"))
print("Quality score:")
print(earlier_checkpoint.values.get("quality_score"))

# Resume from that checkpoint
print("\n===== RESUMING FROM CHECKPOINT =====")

replayed = hitl_graph.invoke(
    None,
    earlier_checkpoint.config
)

# Get the latest state after resuming
replayed_state = hitl_graph.get_state(
    earlier_checkpoint.config
)

print("\n===== REPLAY RESULT =====")
print("Next node after replay:")
print(replayed_state.next)

print("\nFinal answer:")
print(replayed_state.values.get("final_answer"))

print("\nLatest log:")
print(replayed_state.values["log"][-1])

===== PERSISTED THREAD STATE =====
Thread ID: demo-thread-1

Next node:
()

Retry count:
1

Human approval:
False

Final answer:
The agent-framework has made significant progress, including promoting the agent-framework-ag-ui package to stable, adding security guidance for custom MCP Streamable HTTP clients, and fixing issues with stateless replay and Gemini 3 thought signatures. The orchestration layer has reached 1.0, providing stable support for coordination patterns across Python and .NET. New features such as FastAPI SSE keepalive support and forwarding skill directories to the Copilot session have also been added, aiming to improve the framework's stability and functionality.

Latest log entry:
[send_email] REJECTED -> action cancelled, nothing sent


===== CHECKPOINT HISTORY =====
Step 0: next=('__start__',) | retry_count=None | quality_score=None | (no log yet)
Step 1: next=('plan',) | retry_count=0 | quality_score=0 | (no log yet)
Step 2: next=('retrieve',) | retry_count=0 | q

### AgentExecutor vs. LangGraph

| | LangChain `AgentExecutor` | LangGraph |
|---|---|---|
| **Control flow** | A built-in agent execution loop that repeatedly lets the agent reason and call tools | Explicit graph structure designed node-by-node |
| **Branching / loops** | Possible, but more difficult to express as a specific workflow with controlled branches and retry limits | Native via conditional edges; retry caps and self-correction loops are explicit |
| **Pausing for a human** | Requires additional application-level handling | Native interrupt mechanisms such as `interrupt_before` / `interrupt_after`, typically combined with a checkpointer |
| **Persistence** | Requires application-level state/history management | Checkpointers provide graph-state persistence keyed by `thread_id` |
| **Debuggability** | Execution traces/logs can be inspected | State history and checkpoints make workflow state easier to inspect and replay |
| **Best for** | Simpler tool-using agents where a built-in agent loop is sufficient | Stateful, branching, cyclical, human-in-the-loop, or otherwise explicitly controlled workflows |

**Rule of thumb:** Use a simpler agent loop when the task is mostly a single reasoning-and-tool-calling cycle with straightforward control flow. Reach for LangGraph when you need explicit branching, loop-back/retry logic, human approval, checkpointed state, or a workflow that benefits from being represented as a graph.